In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV

In [3]:
df = pd.read_csv("nifty_classification.csv")
df

,Date,close,high,low,open,volume,sma_20,ema_9,MACD_12_26_9,MACDh_12_26_9,...,return,ret_lag_1,ret_lag_2,ret_lag_3,ret_lag_5,roll_std_20,roll_mean_5,india_vix,usdinr_return,target_cls
0,2015-02-20,8833.599609,8899.950195,8816.299805,8895.500000,197600,8773.857422,8789.749445,103.769185,4.583071,...,-0.693627,0.295410,0.678257,0.043718,0.975374,0.770182,0.280443,21.270000,0.146599,0
1,2015-02-23,8754.950195,8869.000000,8736.099609,8856.849609,139600,8773.534912,8782.789595,94.882288,-3.443060,...,-0.890344,-0.693627,0.295410,0.678257,1.078456,0.794944,-0.113317,21.629999,-0.102950,1
2,2015-02-24,8762.099609,8800.500000,8726.750000,8772.900391,149800,8769.859912,8778.651598,87.408664,-8.733347,...,0.081661,-0.890344,-0.693627,0.295410,0.043718,0.770028,-0.105729,21.549999,0.112719,1
3,2015-02-25,8767.250000,8840.650391,8751.400391,8801.900391,135200,8762.697412,8776.371278,80.968008,-12.139203,...,0.058780,0.081661,-0.890344,-0.693627,0.678257,0.741911,-0.229624,20.780001,0.099725,0
4,2015-02-26,8683.849609,8786.049805,8669.450195,8779.000000,217500,8751.174902,8757.866945,68.346175,-19.808829,...,-0.951272,0.058780,0.081661,-0.890344,0.295410,0.766266,-0.478960,20.580000,-0.572048,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2419,2024-12-20,23587.500000,24065.800781,23537.349609,23960.699219,442700,24362.677637,24208.114675,-45.527084,-77.944409,...,-1.520557,-1.021331,-0.563570,-1.346873,0.894555,0.839084,-0.971256,15.070000,0.133748,1
2420,2024-12-23,23753.449219,23869.550781,23647.199219,23738.199219,189800,24339.255078,24117.181584,-82.033872,-91.560957,...,0.703547,-1.520557,-1.021331,-0.563570,-0.403947,0.796137,-0.749757,13.520000,-0.392127,0
2421,2024-12-24,23727.650391,23867.650391,23685.150391,23769.099609,177700,24315.912598,24039.275345,-111.759223,-97.029046,...,-0.108611,0.703547,-1.520557,-1.021331,-1.346873,0.796132,-0.502104,13.180000,0.181176,1
2422,2024-12-26,23750.199219,23854.500000,23653.599609,23775.800781,177700,24289.677539,23981.460120,-131.975939,-93.796610,...,0.095032,-0.108611,0.703547,-1.520557,-0.563570,0.791192,-0.370384,14.040000,0.272237,1


In [4]:
df.isnull().sum()

Date              0
close             0
high              0
low               0
open              0
volume            0
sma_20            0
ema_9             0
MACD_12_26_9      0
MACDh_12_26_9     0
MACDs_12_26_9     0
rsi_14            0
cci_20            0
roc_10            0
BBL_20_2.0_2.0    0
BBM_20_2.0_2.0    0
BBU_20_2.0_2.0    0
BBB_20_2.0_2.0    0
BBP_20_2.0_2.0    0
atr_14            0
obv               0
return            0
ret_lag_1         0
ret_lag_2         0
ret_lag_3         0
ret_lag_5         0
roll_std_20       0
roll_mean_5       0
india_vix         0
usdinr_return     3
target_cls        0
dtype: int64

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2424 entries, 0 to 2423
Data columns (total 31 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            2424 non-null   object 
 1   close           2424 non-null   float64
 2   high            2424 non-null   float64
 3   low             2424 non-null   float64
 4   open            2424 non-null   float64
 5   volume          2424 non-null   int64  
 6   sma_20          2424 non-null   float64
 7   ema_9           2424 non-null   float64
 8   MACD_12_26_9    2424 non-null   float64
 9   MACDh_12_26_9   2424 non-null   float64
 10  MACDs_12_26_9   2424 non-null   float64
 11  rsi_14          2424 non-null   float64
 12  cci_20          2424 non-null   float64
 13  roc_10          2424 non-null   float64
 14  BBL_20_2.0_2.0  2424 non-null   float64
 15  BBM_20_2.0_2.0  2424 non-null   float64
 16  BBU_20_2.0_2.0  2424 non-null   float64
 17  BBB_20_2.0_2.0  2424 non-null   f

In [7]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date').sort_index()

# Basic calendar features
df['day_of_week']  = df.index.dayofweek      # 0=Mon … 4=Fri
df['month']        = df.index.month           # 1–12
df['quarter']      = df.index.quarter          # 1–4
df['week_of_year'] = df.index.isocalendar().week.astype(int)
df['year']         = df.index.year              # trend proxy

# Day-of-month effects (turn-of-month anomaly)
df['day_of_month']   = df.index.day
df['is_month_start'] = df.index.is_month_start.astype(int)
df['is_month_end']   = df.index.is_month_end.astype(int)
df['is_quarter_end'] = df.index.is_quarter_end.astype(int)

In [11]:
df

,close,high,low,open,volume,sma_20,ema_9,MACD_12_26_9,MACDh_12_26_9,MACDs_12_26_9,...,target_cls,day_of_week,month,quarter,week_of_year,year,day_of_month,is_month_start,is_month_end,is_quarter_end
Date,,,,,,,,,,,,,,,,,,,,,
2015-02-20,8833.599609,8899.950195,8816.299805,8895.500000,197600,8773.857422,8789.749445,103.769185,4.583071,99.186113,...,0,4,2,1,8,2015,20,0,0,0
2015-02-23,8754.950195,8869.000000,8736.099609,8856.849609,139600,8773.534912,8782.789595,94.882288,-3.443060,98.325348,...,1,0,2,1,9,2015,23,0,0,0
2015-02-24,8762.099609,8800.500000,8726.750000,8772.900391,149800,8769.859912,8778.651598,87.408664,-8.733347,96.142012,...,1,1,2,1,9,2015,24,0,0,0
2015-02-25,8767.250000,8840.650391,8751.400391,8801.900391,135200,8762.697412,8776.371278,80.968008,-12.139203,93.107211,...,0,2,2,1,9,2015,25,0,0,0
2015-02-26,8683.849609,8786.049805,8669.450195,8779.000000,217500,8751.174902,8757.866945,68.346175,-19.808829,88.155004,...,1,3,2,1,9,2015,26,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-20,23587.500000,24065.800781,23537.349609,23960.699219,442700,24362.677637,24208.114675,-45.527084,-77.944409,32.417324,...,1,4,12,4,51,2024,20,0,0,0
2024-12-23,23753.449219,23869.550781,23647.199219,23738.199219,189800,24339.255078,24117.181584,-82.033872,-91.560957,9.527085,...,0,0,12,4,52,2024,23,0,0,0
2024-12-24,23727.650391,23867.650391,23685.150391,23769.099609,177700,24315.912598,24039.275345,-111.759223,-97.029046,-14.730176,...,1,1,12,4,52,2024,24,0,0,0


In [33]:
print(df[df["target_cls"] == 1].shape[0], "\n", df[df["target_cls"] == 0].shape[0])

1308 
 1116


## Feature Engineering

In [35]:
# Volatility regime bucket
df['vol_regime'] = pd.qcut(df['roll_std_20'], q=3, labels=[0,1,2])

In [36]:
# RSI state (don't use raw RSI — discretize it)
df['rsi_zone'] = pd.cut(df['rsi_14'],
    bins=[0,30,50,70,100], labels=[0,1,2,3])

In [38]:
# RSI in a high-volatility regime behaves differently
df['rsi_x_vol'] = df['rsi_14'] * df['roll_std_20']

In [39]:
# Volume confirmation of price move
df['vol_price_confirm'] = (
    (df['return'] > 0) & (df['volume'] > df['volume'].rolling(20).mean())
).astype(int)

In [41]:
# Bollinger squeeze + breakout
df['bb_width'] = (df['BBU_20_2.0_2.0'] - df['BBL_20_2.0_2.0']) / df['BBM_20_2.0_2.0']
df['bb_squeeze'] = (df['bb_width'] < df['bb_width'].rolling(20).quantile(0.2)).astype(int)

In [42]:
# How many days in a row has it gone up? Reversal signal.
def streak(series):
    streaks = []
    count = 0
    for val in series:
        if val > 0: count = max(1, count + 1)
        elif val < 0: count = min(-1, count - 1)
        else: count = 0
        streaks.append(count)
    return streaks

df['price_streak'] = streak(df['return'])
# +5 = 5 consecutive up days (mean-reversion signal for classification)

In [43]:
df['close_position'] = (df['close'] - df['low']) / (df['high'] - df['low'])
# 1.0 = closed at top of range (bullish), 0.0 = closed at bottom (bearish)

In [44]:
df['vol_zscore'] = (
    (df['volume'] - df['volume'].rolling(20).mean()) /
    df['volume'].rolling(20).std()
)
# Abnormal volume often precedes directional moves

In [51]:
# OBV is a CUMULATIVE sum — it encodes the entire price history level
# This is a subtle data leakage vector in walk-forward validation
# Replace it with:
df['obv_change'] = df['obv'].pct_change()   # rate of change only
df.drop(columns=['obv'], inplace=True)

In [52]:
# 1. Raw date (already your index)
df.drop(columns=['date'], errors='ignore', inplace=True)

# 2. Raw OHLC prices — not prices, but their LEVELS
# Models learn "Nifty at 22000 = up" which is nonsense
drop_levels = ['open', 'high', 'low', 'close', 'adj_close']
df.drop(columns=drop_levels, errors='ignore', inplace=True)

# 3. Raw volume level (keep z-score version, drop raw)
df.drop(columns=['volume'], errors='ignore', inplace=True)

# 4. Intermediate calculation columns (not features themselves)
drop_intermediates = [
    'BBM_20_2.0_2,0',   # Bollinger mid — same as SMA20, redundant
    'sma_20',       # if you already have close_vs_sma20 ratio
    'ema_9',        # if you already have ema_9_vs_ema_21 crossover flag
]
df.drop(columns=drop_intermediates, errors='ignore', inplace=True)

In [61]:
# year = 2015, 2016 ... 2024
# this creates misleadingly high split values
# Replace with years_since_start instead:
df['years_since_start'] = df.index.year - df.index.year.min()

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2424 entries, 2015-02-20 to 2024-12-27
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   MACD_12_26_9       2424 non-null   float64 
 1   MACDh_12_26_9      2424 non-null   float64 
 2   MACDs_12_26_9      2424 non-null   float64 
 3   rsi_14             2424 non-null   float64 
 4   cci_20             2424 non-null   float64 
 5   roc_10             2424 non-null   float64 
 6   BBL_20_2.0_2.0     2424 non-null   float64 
 7   BBM_20_2.0_2.0     2424 non-null   float64 
 8   BBU_20_2.0_2.0     2424 non-null   float64 
 9   BBB_20_2.0_2.0     2424 non-null   float64 
 10  BBP_20_2.0_2.0     2424 non-null   float64 
 11  atr_14             2424 non-null   float64 
 12  return             2424 non-null   float64 
 13  ret_lag_1          2424 non-null   float64 
 14  ret_lag_2          2424 non-null   float64 
 15  ret_lag_3          2424 non-null   fl

In [56]:
df.isnull().sum()

MACD_12_26_9          0
MACDh_12_26_9         0
MACDs_12_26_9         0
rsi_14                0
cci_20                0
roc_10                0
BBL_20_2.0_2.0        0
BBM_20_2.0_2.0        0
BBU_20_2.0_2.0        0
BBB_20_2.0_2.0        0
BBP_20_2.0_2.0        0
atr_14                0
return                0
ret_lag_1             0
ret_lag_2             0
ret_lag_3             0
ret_lag_5             0
roll_std_20           0
roll_mean_5           0
india_vix             0
usdinr_return         3
target_cls            0
day_of_week           0
month                 0
quarter               0
week_of_year          0
year                  0
day_of_month          0
is_month_start        0
is_month_end          0
is_quarter_end        0
vol_regime            0
rsi_zone              0
rsi_x_vol             0
vol_price_confirm     0
bb_width              0
bb_squeeze            0
price_streak          0
close_position        0
vol_zscore           19
obv_change            1
dtype: int64

In [62]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns="target_cls"), df.target_cls, test_size=0.2, random_state=42)

In [63]:
mean_imputer = SimpleImputer(strategy="mean")

X_train["usdinr_return"] = mean_imputer.fit_transform(X_train[["usdinr_return"]])
X_test["usdinr_return"] = mean_imputer.transform(X_test[["usdinr_return"]])

X_train["vol_zscore"] = mean_imputer.fit_transform(X_train[["vol_zscore"]])
X_test["vol_zscore"] = mean_imputer.transform(X_test[["vol_zscore"]])

X_train["obv_change"] = mean_imputer.fit_transform(X_train[["obv_change"]])
X_test["obv_change"] = mean_imputer.transform(X_test[["obv_change"]])

In [64]:
X_train.shape

(1939, 41)

In [60]:
# Tree-based models are scale-invariant.

In [65]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train only
X_test_scaled  = scaler.transform(X_test)       # apply to test

In [ ]:
# Model for Scaled Data
# Later